In [1]:
import pandas as pd
import numpy as np

mil = pd.read_csv("C:/Users/강재원/Desktop/dash/current_usd.csv")
gdp = pd.read_csv("C:/Users/강재원/Desktop/dash/gdp_calculated.csv")
hazard = pd.read_csv("C:/Users/강재원/Desktop/dash/human_hazard_trend.csv")
tiv = pd.read_csv("C:/Users/강재원/Desktop/dash/tiv_5y_share.csv")

In [2]:
years = [str(year) for year in range(2000, 2026)]

In [3]:
mil_long = mil.melt(
    id_vars="Country",
    value_vars=years,
    var_name="Year",
    value_name="Military_Expenditure"
)

mil_long["Year"] = mil_long["Year"].astype(int)

mil_long.head()

,Country,Year,Military_Expenditure
0,Algeria,2000,1881.163649
1,Libya,2000,1085.534665
2,Morocco,2000,859.151483
3,Tunisia,2000,332.389763
4,Angola,2000,583.621334


In [4]:
gdp_long = gdp.melt(
    id_vars="Country",
    value_vars=years,
    var_name="Year",
    value_name="GDP"
)

gdp_long["Year"] = gdp_long["Year"].astype(int)

gdp_long.head()

,Country,Year,GDP
0,Algeria,2000,54790.206724
1,Libya,2000,39497.061637
2,Morocco,2000,38858.714116
3,Tunisia,2000,21473.538137
4,Angola,2000,9129.634978


In [5]:
country_code_map = (
    tiv[["Country", "Country_Code"]]
    .drop_duplicates()
    .set_index("Country")["Country_Code"]
    .to_dict()
)

In [6]:
manual_country_codes = {
    "Bosnia and Herzegovina": "BIH",
    "Cape Verde": "CPV",
    "Congo, DR": "COD",
    "Congo, Republic": "COG",
    "Eswatini": "SWZ",
    "Gambia, The": "GMB",
    "Guinea-Bissau": "GNB",
    "Korea, North": "PRK",
    "Korea, South": "KOR",
    "Kyrgyz Republic": "KGZ",
    "Panama": "PAN",
    "Timor Leste": "TLS",
    "Türkiye": "TUR",
    "United States of America": "USA",
    "Kosovo": "XKX",
    "Yemen": "YEM"
}

country_code_map.update(manual_country_codes)

In [7]:
mil_long["Iso3"] = mil_long["Country"].map(country_code_map)
gdp_long["Iso3"] = gdp_long["Country"].map(country_code_map)

In [8]:
# 국가코드가 없는 국가들(전부 ~ 1992년까지 존재했던 국가들이라 2000년부터 기간을 잡은 우리는 삭제해도 괜찮음)
print(
    mil_long.loc[
        mil_long["Iso3"].isna(),
        "Country"
    ].drop_duplicates().tolist()
)

['Czechoslovakia', 'German Democratic Republic', 'Yugoslavia', 'USSR', 'Yemen, North']


In [9]:
# 위의 국가들 삭제
mil_long = mil_long.dropna(subset=["Iso3"]).copy()
gdp_long = gdp_long.dropna(subset=["Iso3"]).copy()

In [10]:
# 분쟁데이터 컬럼명 year로 변경, 정수화, 사용할 컬럼만 추출, year는 2016~2025
hazard_clean = hazard.rename(
    columns={"Data_reference_year": "Year"}
).copy()

hazard_clean["Year"] = hazard_clean["Year"].astype(int)

hazard_clean = hazard_clean[
    [
        "Iso3",
        "Year",
        "IndicatorScore"
    ]
]

hazard_clean.head()

,Iso3,Year,IndicatorScore
0,AFG,2022,9.9
1,AGO,2022,7.4
2,ALB,2022,1.0
3,ARE,2022,2.6
4,ARG,2022,1.1


In [11]:
# 컬럼명 국가코드를 Iso3로 변경
tiv_clean = tiv.rename(
    columns={"Country_Code": "Iso3"}
).copy()

tiv_clean["Year"] = tiv_clean["Year"].astype(int)

In [12]:
tiv_clean = tiv_clean[
    [
        "Iso3",
        "Country",
        "Region",
        "Year",
        "Window",
        "TIV_5Y_Sum",
        "World_TIV_5Y_Sum",
        "TIV_5Y_Share"
    ]
]

In [13]:
# 군사비 + GDP를 ISO3 + Year로 merge
merged = pd.merge(
    mil_long[
        [
            "Country",
            "Iso3",
            "Year",
            "Military_Expenditure"
        ]
    ],
    gdp_long[
        [
            "Iso3",
            "Year",
            "GDP"
        ]
    ],
    on=["Iso3", "Year"],
    how="outer"
)

merged.head()

,Country,Iso3,Year,Military_Expenditure,GDP
0,Afghanistan,AFG,2000,NaN,NaN
1,Afghanistan,AFG,2001,NaN,NaN
2,Afghanistan,AFG,2002,NaN,NaN
3,Afghanistan,AFG,2003,NaN,NaN
4,Afghanistan,AFG,2004,125.111557,5145.96864


In [14]:
# TIV merge
merged = pd.merge(
    merged,
    tiv_clean,
    on=["Iso3", "Year"],
    how="outer",
    suffixes=("", "_tiv")
)

In [15]:
# 분쟁위험도 merge
merged = pd.merge(
    merged,
    hazard_clean,
    on=["Iso3", "Year"],
    how="outer"
)

In [16]:
# 2000~ 2025
merged = merged[
    (merged["Year"] >= 2000) &
    (merged["Year"] <= 2025)
].copy()

In [17]:
# TIV에 있는 Country_tiv때문에 국가명 컬럼이 중복될 수 있음
# 기존 국가명이 있으면 국가명을 쓰고, 없으면 TIV 국가명 사용
# 사용 후 Country_tiv 컬럼 삭제
merged["Country"] = merged["Country"].fillna(
    merged["Country_tiv"]
)

merged = merged.drop(
    columns=["Country_tiv"]
)

In [18]:
column_order = [
    "Country",
    "Iso3",
    "Region",
    "Year",
    "Military_Expenditure",
    "GDP",
    "IndicatorScore",
    "TIV_5Y_Share",
    "TIV_5Y_Sum",
    "World_TIV_5Y_Sum",
    "Window"
]

merged = merged[column_order]

In [19]:
merged = merged.sort_values(
    by=["Country", "Year"]
).reset_index(drop=True)

In [20]:
duplicates = merged[
    merged.duplicated(
        subset=["Iso3", "Year"],
        keep=False
    )
]

print("중복 행 수:", len(duplicates))

중복 행 수: 0


In [21]:
merged[
    [
        "Military_Expenditure",
        "GDP",
        "IndicatorScore",
        "TIV_5Y_Share"
    ]
].isna().sum()

Military_Expenditure     767
GDP                      861
IndicatorScore          2862
TIV_5Y_Share             856
dtype: int64

In [22]:
merged = merged.rename(
    columns={"Military_Expenditure": "current_usd"}
)

In [23]:
print(merged.shape)
print(merged["Iso3"].nunique())
print(merged["Year"].min())
print(merged["Year"].max())

display(merged.head(20))

(4772, 11)
194
2000
2025


,Country,Iso3,Region,Year,current_usd,GDP,IndicatorScore,TIV_5Y_Share,TIV_5Y_Sum,World_TIV_5Y_Sum,Window
0,Afghanistan,AFG,NaN,2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Afghanistan,AFG,NaN,2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,NaN,2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Afghanistan,AFG,NaN,2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Afghanistan,AFG,Asia,2004,125.111557,5145.968640,NaN,0.035532,34.40,96813.81,2000-2004
5,Afghanistan,AFG,Asia,2005,122.727193,6160.800481,NaN,0.069413,67.80,97676.36,2001-2005
6,Afghanistan,AFG,Asia,2006,131.346231,6926.688981,NaN,0.068554,71.16,103801.38,2002-2006
7,Afghanistan,AFG,Asia,2007,219.580214,8556.404465,NaN,0.068110,76.72,112642.01,2003-2007
8,Afghanistan,AFG,Asia,2008,240.532595,10298.774904,NaN,0.190829,224.41,117597.49,2004-2008
9,Afghanistan,AFG,Asia,2009,251.869515,12066.110960,NaN,0.457686,550.68,120318.36,2005-2009


In [24]:
merged = merged.drop(
    columns=["World_TIV_5Y_Sum", "Window", "Region"]
)

In [25]:
hazard_score = pd.read_csv("C:/Users/강재원/Desktop/integrate.csv")

# Kosovo 국가코드 수정
hazard_score.loc[
    hazard_score["Country"] == "Kosovo",
    "Iso3"
] = "XKX"

# 필요한 컬럼만 선택
hazard_score = hazard_score[
    ["Iso3", "Year", "human_hazard_score","share_gdp"]
]

# 중복 제거
hazard_score = hazard_score.drop_duplicates(
    subset=["Iso3", "Year"]
)

# merged에 추가
merged = pd.merge(
    merged,
    hazard_score,
    on=["Iso3", "Year"],
    how="left"
)

In [26]:
merged = merged.drop(
    columns=["IndicatorScore"]
)

In [27]:
merged = merged.rename(
    columns={"GDP": "gdp_calculated"}
)

In [29]:
merged.to_csv(
    "Integrate_new.csv",
    index=False,
    encoding="utf-8-sig"
)